<a href="https://colab.research.google.com/github/maps-05/portfolio/blob/main/AssociationRuleMining_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Association Rule Mining with Apriori

## Notebook Summary

This notebook performs Association Rule Mining using the Apriori algorithm on a dataset containing various applicant attributes. It aims to discover frequent itemsets and derive association rules, particularly focusing on what makes an applicant 'Employable'. The process includes data loading, preprocessing with `TransactionEncoder`, applying the `apriori` algorithm to find frequent itemsets, and then generating `association_rules` to analyze relationships between items. The notebook also addresses specific questions regarding the least supported itemsets and the strength of associations between certain attributes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:

# Load Dataset

df = pd.read_csv('https://raw.githubusercontent.com/renatomaaliw3/public_files/refs/heads/master/Data%20Sets/Apriori%20Data.csv')
df.head()

,Technical Skills,Professional Certifications,Communication Skills,Grit,Work Habits,Employability
0,Technical Skills - Fair,Professional Certifications - 1,Communication Skills - 3,Grit - 1,Work Habits - 3,Less Employable
1,Technical Skills - Fair,Professional Certifications - 1,Communication Skills - 4,Grit - 2,Work Habits - 4,Employable
2,Technical Skills - Very Good,Professional Certifications - 2,Communication Skills - 5,Grit - 2,Work Habits - 4,Employable
3,Technical Skills - Fair,Professional Certifications - 2,Communication Skills - 4,Grit - 2,Work Habits - 3,Employable
4,Technical Skills - Good,Professional Certifications - 1,Communication Skills - 4,Grit - 2,Work Habits - 4,Employable


In [ ]:
# Data Preprocessing

from mlxtend.preprocessing import TransactionEncoder

transactions = df.apply(lambda row: row.dropna().tolist(), axis = 1).tolist()

encoder = TransactionEncoder()

transaction_matrix = encoder.fit_transform(transactions)

transaction_df = pd.DataFrame(transaction_matrix, columns = encoder.columns_)
transaction_df

,Communication Skills - 2,Communication Skills - 3,Communication Skills - 4,Communication Skills - 5,Employable,Grit - 1,Grit - 2,Less Employable,Professional Certifications - 1,Professional Certifications - 2,Technical Skills - Excellent,Technical Skills - Fair,Technical Skills - Good,Technical Skills - Pass,Technical Skills - Very Good,Underemployed,Work Habits - 3,Work Habits - 4,Work Habits - 5
0,False,True,False,False,False,True,False,True,True,False,False,True,False,False,False,False,True,False,False
1,False,False,True,False,True,False,True,False,True,False,False,True,False,False,False,False,False,True,False
2,False,False,False,True,True,False,True,False,False,True,False,False,False,False,True,False,False,True,False
3,False,False,True,False,True,False,True,False,False,True,False,True,False,False,False,False,True,False,False
4,False,False,True,False,True,False,True,False,True,False,False,False,True,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,False,False,False,True,True,False,True,False,False,True,False,True,False,False,False,False,False,False,True
496,False,False,False,True,False,True,False,False,True,False,False,True,False,False,False,True,True,False,False
497,False,False,True,False,True,False,True,False,False,True,True,False,False,False,False,False,True,False,False
498,False,False,False,True,False,True,False,False,True,False,False,False,True,False,False,True,False,True,False


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# Apply the Apriori algorithm
frequent_itemsets = apriori(transaction_df, min_support = 0.2, use_colnames = True)

frequent_itemsets

,support,itemsets
0,0.486,(Communication Skills - 4)
1,0.332,(Communication Skills - 5)
2,0.546,(Employable)
3,0.478,(Grit - 1)
4,0.522,(Grit - 2)
5,0.520,(Professional Certifications - 1)
6,0.480,(Professional Certifications - 2)
7,0.542,(Technical Skills - Fair)
8,0.352,(Technical Skills - Good)
9,0.294,(Underemployed)


In [ ]:
# Q17: What itemset/s has the least support?

least_support = frequent_itemsets[frequent_itemsets['support'] == frequent_itemsets['support'].min()]
least_support

,support,itemsets
46,0.202,"(Grit - 2, Employable, Work Habits - 4)"


In [ ]:
pd.set_option('display.max_columns', 100)

rules = association_rules(frequent_itemsets, num_itemsets = len(transaction_df), metric = "confidence", min_threshold = 0.2)
rules.loc[:, :'lift']

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,(Communication Skills - 4),(Employable),0.486,0.546,0.268,0.551440,1.009964
1,(Employable),(Communication Skills - 4),0.546,0.486,0.268,0.490842,1.009964
2,(Communication Skills - 4),(Grit - 1),0.486,0.478,0.232,0.477366,0.998674
3,(Grit - 1),(Communication Skills - 4),0.478,0.486,0.232,0.485356,0.998674
4,(Communication Skills - 4),(Grit - 2),0.486,0.522,0.254,0.522634,1.001214
...,...,...,...,...,...,...,...
199,"(Technical Skills - Fair, Grit - 1)","(Professional Certifications - 1, Underemployed)",0.396,0.294,0.238,0.601010,2.044252
200,(Underemployed),"(Technical Skills - Fair, Professional Certifi...",0.294,0.392,0.238,0.809524,2.065112
201,(Professional Certifications - 1),"(Technical Skills - Fair, Underemployed, Grit ...",0.520,0.238,0.238,0.457692,1.923077
202,(Grit - 1),"(Professional Certifications - 1, Underemploye...",0.478,0.238,0.238,0.497908,2.092050


In [ ]:
# Q18: According to Association Rules, what is the most superior rule that makes an applicant 'Employable'?

employable_rules = rules[rules['consequents'].apply(lambda x: 'Employable' in x)]
sup_rule = employable_rules.sort_values('confidence', ascending=False).head(1)
sup_rule

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
54,"(Communication Skills - 4, Professional Certif...",(Employable),0.234,0.546,0.234,1.0,1.831502,1.0,0.106236,inf,0.592689,0.428571,1.0,0.714286


In [ ]:
sup_rule['antecedents'].iloc[0], sup_rule['consequents'].iloc[0]

(frozenset({'Communication Skills - 4', 'Professional Certifications - 2'}),
 frozenset({'Employable'}))

In [ ]:
# Q19: Does {'Underemployed'} and {'Grit-1'} have strong assocation? Explain

rules[(rules['antecedents'] == {'Underemployed'}) &
            (rules['consequents'] == {'Grit-1'})]


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [ ]:
# Q20: Does {'Grit - 2', 'Technical Skills - Good'} makes you {'Employable'}? Explain

rules[(rules['antecedents'] == {'Grit - 2', 'Technical Skills - Good'}) &
            (rules['consequents'] == {'Employable'})]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
100,"(Grit - 2, Technical Skills - Good)",(Employable),0.292,0.546,0.292,1.0,1.831502,1.0,0.132568,inf,0.641243,0.534799,1.0,0.767399
